In [5]:
# 1. Reset directory and clone cleanly
%cd /content
!rm -rf Baseera
!git clone https://github.com/Hager-ali191/Baseera.git
%cd /content/Baseera

# 2. Install required packages
!pip install -q -r backend/requirements.txt
!pip install -q -r NiceGUI/requirements.txt pillow

# 3. Patch NiceGUI/app.py to ensure 'import os' is present
app_path = "NiceGUI/app.py"
with open(app_path, "r") as f:
    content = f.read()

if "import os" not in content:
    with open(app_path, "w") as f:
        f.write("import os\n" + content)
    print("Updated NiceGUI/app.py with 'import os'")

/content
Cloning into 'Baseera'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 142 (delta 42), reused 111 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 25.75 MiB | 19.29 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/Baseera
Updated NiceGUI/app.py with 'import os'
Successfully mounted NiceGUI into backend/main.py


In [6]:
main_py_path = "backend/main.py"
with open(main_py_path, "r") as f:
    main_content = f.read()

if "ui.run_with" not in main_content:
    mount_code = """

# --- NiceGUI Integration ---
import sys, os
sys.path.insert(0, os.path.abspath("../NiceGUI"))
from nicegui import ui
import pages.home, pages.about, pages.demo

ui.run_with(app, storage_secret="your_unique_secret_key")
"""
    with open(main_py_path, "a") as f:
        f.write(mount_code)
    print("Backend patched cleanly!")

In [7]:
import subprocess
import time
import requests
import os
import gc
import torch

# Clean up memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Kill any existing process on port 8000
!fuser -k 8000/tcp || true
time.sleep(1)

# Configure environment path
env = os.environ.copy()
env["PYTHONPATH"] = f"{os.getcwd()}/backend:{os.getcwd()}/NiceGUI:" + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"

# Launch combined server
log_file = open("app.log", "w")
server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    cwd="backend",
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Starting unified server on port 8000...")
time.sleep(6)

# Verify health status
if server.poll() is None:
    print("Combined FastAPI + NiceGUI server is UP on port 8000!")
else:
    print(f"Server failed to start (exit code {server.poll()}). Logs:")
    !cat app.log

8000/tcp:              876
Starting unified server on port 8000...
Combined FastAPI + NiceGUI server is UP on port 8000!


In [8]:
import subprocess
import time
import re

# Download cloudflared binary if not present
!if [ ! -f cloudflared ]; then wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared; fi

# Terminate old tunnel processes
!pkill cloudflared || true
time.sleep(1)

# Start Cloudflare tunnel targeting port 8000
log_file = open("tunnel.log", "w")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Establishing Cloudflare tunnel...")
time.sleep(8)

# Read and print live link
with open("tunnel.log", "r") as f:
    logs = f.read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if match:
        print("\n=======================================================")
        print("YOUR LIVE PROJECT URL IS:")
        print(match.group(0))
        print("=======================================================\n")
    else:
        print("Tunnel starting... Run '!cat tunnel.log' if URL doesn't appear.")

Establishing Cloudflare tunnel...
Tunnel starting... Run '!cat tunnel.log' if URL doesn't appear.


In [9]:
!cat app.log
!cat tunnel.log

[Baseera] Capping CPU inference to 1 thread(s) (of 2 available) — set BASEERA_CPU_THREADS to change this.
[Baseera] Loading Whisper (speech-to-text)...
[Baseera] Loading YOLO detection models...
[Baseera] Loading local instruct LLM (Qwen/Qwen2.5-0.5B-Instruct)...
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 321.10it/s]
[Baseera] All models loaded: ['yolov8n', 'yolov8s']
INFO:     Started server process [1694]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
2026-09-22T18:04:20Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such